### Phase 3: Modellierung (Der Vergleich)

Hier werden die konkurrierenden Modelle implementiert:

#### 1. Markov-Switching-Modell (MSM)
*   **Library:** `statsmodels.tsa.regime_switching.markov_regression`
*   **Logik:** Ein statistisches Modell, das Wahrscheinlichkeiten für Regimes berechnet.

#### 2. Hidden-Markov-Models (HMM)
*   **Library:** `hmmlearn.hmm`
*   **Logik:** Unsupervised Learning (Clustering), das Zeitabschnitte mit ähnlichen statistischen Verteilungen gruppiert, um verborgene Marktregimes zu identifizieren.

#### 3. LSTM-Netzwerk
*   **Library:** `TensorFlow/Keras` oder `PyTorch`.
*   **Architektur:**
    *   Input: Zeitreihen-Fenster (z.B. die letzten 60 Tage der Features).
    *   Layer: LSTM-Layer -> Dropout -> Dense (Softmax).
    
#### 4. Transformer-Netzwerk
*   **Library:** `PyTorch` (`torch.nn.TransformerEncoder`)
*   **Architektur:**
    *   Input: Zeitreihen-Fenster (z.B. die letzten 60 Tage der Features).
    *   Layer: Linear Projection → Positional Encoding → TransformerEncoder (Multi-Head Self-Attention) → Dense (Sigmoid).
*   **Zweck:** Test von Hypothese H2 — Attention-Mechanismus vs. ökonometrische MSM.

Modelle die ein Feedback (gelabelte Daten) benötigen, um Regime zu erkennen, erhalten diese durch das genauste Modell (im Projektverlauf ermittelt) -> Aktuell: Markov-Switching (Univariat)

In [ ]:
# --- Central Config ---
import sys; sys.path.insert(0, "../config")
from config_loader import cfg

In [ ]:
# --- Regime Signal Validation (Helper) ---
# --- Gemeinsame Funktionen aus src/ ---
sys.path.insert(0, "..")
from src.models.common import BEAR_REGIME, BULL_REGIME, validate_regime_signal, create_sequences

# --- Sanity Check ---
# Wird zu Ende jeder nachfolgenden Model-Cell aufgerufen
def validate_regime_signal(data, model_name: str, auto_invert: bool = True) -> None:
    """
    Standardized sanity check for regime signals.
    Expects {model_name}_Prob and {model_name}_Signal in data.
    """
    prob_col = f'{model_name}_Prob'
    signal_col = f'{model_name}_Signal'
    stats_cols = ['Returns', 'VIX', 'Yield_Spread', prob_col]

    # Regime Statistics
    print(f"\n{'='*60}")
    print(f"   {model_name} — Regime-Statistik")
    print(f"{'='*60}")
    available = [c for c in stats_cols if c in data.columns]
    print(data.groupby(signal_col)[available].mean())
    print(f"\nSignal-Verteilung:\n{data[signal_col].value_counts()}")

    # Plausibility Check
    mean_returns = data.groupby(signal_col)['Returns'].mean()
    if mean_returns.get(BEAR_REGIME, 0) > mean_returns.get(BULL_REGIME, 0):
        print(f"\n WARNUNG: {model_name} Bear-Regime ({BEAR_REGIME}) "
              f"hat höhere Returns als Bull ({BULL_REGIME})!")
        if auto_invert:
            print("    → Labels könnten vertauscht sein. Invertiere:")
            data[signal_col] = 1 - data[signal_col]
            data[prob_col] = 1 - data[prob_col]
            print("    → Labels automatisch invertiert.")
    else:
        print(f"\n{model_name} Plausibilitäts-Check bestanden.")

    # Validierung
    assert prob_col in data.columns, f"{prob_col} fehlt!"
    assert signal_col in data.columns, f"{signal_col} fehlt!"
    assert data[signal_col].isin([BULL_REGIME, BEAR_REGIME]).all(), "Signal enthält ungültige Werte!"
    assert data[signal_col].isna().sum() == 0, "NaN im Signal!"
    assert data[prob_col].between(0, 1).all(), "Prob außerhalb [0,1]!"
    print("Alle formalen Prüfungen bestanden.")

In [ ]:
import pandas as pd

# Daten aus dem data-Ordner laden
df = pd.read_parquet(cfg.data_path("feature_engineered"))

In [ ]:
# =============================================================================
# 1. Markov-Switching-Modelle (MS Univariate)
# Typ: Ökonometrie (Regression)
# Beschreibung: Zustandsabhängige Regressionsmodelle mit switching variance.
#               MS Univariate: Nur Returns.
# Config-Key: models.msm
# =============================================================================

import matplotlib.pyplot as plt
import warnings
import os

# --- MSM aus src/ ---
from src.models.msm import train_msm, load_msm, predict_msm

# Warnung ignorieren
warnings.filterwarnings("ignore")

# --- 0. Hyperparameter aus zentraler Config laden ---
msm_cfg = cfg.models.msm
persist = cfg.model_persistence
model_file = cfg.model_path("msm")

# --- 1. Vorbereitung: Index auf Business Days setzen ---
df.index = pd.DatetimeIndex(df.index).to_period('B')

MODEL_NAME = "MSM"

if persist.enabled and os.path.exists(model_file):
    # ================================================================
    # MODUS A: Gespeichertes Modell laden (Training überspringen)
    # ================================================================
    print(f"{MODEL_NAME}: Lade persistiertes Modell aus {model_file}")
    ms_uni_results = load_msm(
        model_file=model_file,
        returns=df['Returns'],
        k_regimes=msm_cfg.k_regimes,
        switching_variance=msm_cfg.switching_variance,
    )
else:
    # ================================================================
    # MODUS B: Normal trainieren + Modell speichern
    # ================================================================
    print(f"{MODEL_NAME}: Starte Training...")
    ms_uni_results = train_msm(
        returns=df['Returns'],
        k_regimes=msm_cfg.k_regimes,
        switching_variance=msm_cfg.switching_variance,
        model_file=model_file,
    )

# --- Wahrscheinlichkeiten und Signal ableiten ---
df[f'{MODEL_NAME}_Prob'], df[f'{MODEL_NAME}_Signal'] = predict_msm(
    ms_results=ms_uni_results,
    threshold=msm_cfg.threshold,
)

# --- ABSCHLUSS ---
# Index wieder zurück in normales Datetime-Format für Plotting
df.index = df.index.to_timestamp()

print("Markov-Switching-Modell erfolgreich berechnet.")

# --- VISUALISIERUNG ---
model_color = cfg.color_map.get(MODEL_NAME, 'tab:blue')
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(15, 8), sharex=True)

# Plot Wahrscheinlichkeiten
ax1.plot(df.index, df[f'{MODEL_NAME}_Prob'], label='MSM', alpha=0.7, color=model_color)
ax1.set_title("MS Bärenmarkt-Wahrscheinlichkeiten")
ax1.legend()

# Plot Signale
ax2.fill_between(df.index, 0, df[f'{MODEL_NAME}_Signal'], alpha=0.3, label='Signal', color=model_color)
ax2.set_title("MS Handelssignale (1 = Bear/Cash, 0 = Bull/Investiert)")
ax2.legend()

plt.tight_layout()
# Markov-Modelle persistieren
plt.savefig(cfg.asset_path("markov_model"), dpi=300, bbox_inches='tight')
plt.show()

# Kurzer Blick auf das Ergebnis
print(df)

# --- Sanity Check ---
validate_regime_signal(df, MODEL_NAME)

In [ ]:
# =============================================================================
# 2. Hidden Markov Model (HMM)
# Typ: Unsupervised (Clustering)
# Beschreibung: Identifiziert Regime-Cluster über Gaussian-Emissions in
#               Returns, VIX und Yield_Spread ohne gelabelte Daten.
# Config-Key: models.hmm
# =============================================================================

import matplotlib.pyplot as plt
import os

# --- HMM aus src/ ---
from src.models.hmm import train_hmm, load_hmm, predict_hmm

MODEL_NAME = "HMM"

# --- 0. Hyperparameter aus zentraler Config laden ---
hmm_cfg = cfg.models.hmm
persist = cfg.model_persistence
model_file = cfg.model_path("hmm")
scaler_file = cfg.model_path("scaler_hmm")

# --- 1. Features vorbereiten ---
# Returns (Performance), VIX (Angst) und Yield_Spread (Makro)
hmm_features = hmm_cfg.features

if persist.enabled and os.path.exists(model_file) and os.path.exists(scaler_file):
    # ================================================================
    # MODUS A: Gespeichertes Modell laden (Training überspringen)
    # ================================================================
    print(f"{MODEL_NAME}: Lade persistiertes Modell aus {model_file}")
    model_hmm, scaler_hmm, X_hmm_scaled = load_hmm(
        features_df=df[hmm_features],
        model_file=model_file,
        scaler_file=scaler_file,
    )
else:
    # ================================================================
    # MODUS B: Normal trainieren + Modell speichern
    # ================================================================
    print(f"{MODEL_NAME}: Starte Training...")
    model_hmm, scaler_hmm, X_hmm_scaled = train_hmm(
        features_df=df[hmm_features],
        n_components=hmm_cfg.n_components,
        covariance_type=hmm_cfg.covariance_type,
        n_iter=hmm_cfg.n_iter,
        random_state=hmm_cfg.random_state,
        model_file=model_file,
        scaler_file=scaler_file,
    )

# --- Wahrscheinlichkeiten und Signal ableiten ---
df[f'{MODEL_NAME}_Prob'], df[f'{MODEL_NAME}_Signal'] = predict_hmm(
    model=model_hmm,
    X_scaled=X_hmm_scaled,
    returns=df['Returns'],
    threshold=hmm_cfg.threshold,
)

# --- Visualisierung ---
model_color = cfg.color_map.get(MODEL_NAME, 'tab:purple')

plt.figure(figsize=(15, 3))
plt.fill_between(df.index, 0, 1, where=(df[f'{MODEL_NAME}_Signal'] == 1),
                 color=model_color, alpha=0.3, label='Bear Market (HMM)')
plt.plot(df.index, df[f'{MODEL_NAME}_Prob'], color='black', alpha=0.2, label='Bear Prob')
plt.title(f"{MODEL_NAME} Regimes")
plt.legend()
# HMM Regimes persistieren
plt.savefig(cfg.asset_path("hmm_regimes"), dpi=300, bbox_inches='tight')
plt.show()

# --- Sanity Check ---
validate_regime_signal(df, MODEL_NAME)

In [ ]:
# =============================================================================
# 3. LSTM-Netzwerk (Supervised Regime Classification)
# Typ: Supervised (Labels von MS_Univariate)
# Beschreibung: LSTM-Netzwerk mit rollendem Fenster für zeitreihenbasierte
#               Regime-Klassifikation. Labels stammen aus dem MS_Univariate-Modell.
# Config-Key: models.lstm
# =============================================================================

import numpy as np
import matplotlib.pyplot as plt
import os

# --- LSTM aus src/ ---
from src.models.lstm import train_lstm, load_lstm_model, predict_lstm

MODEL_NAME = "LSTM"

# --- 0. Hyperparameter aus zentraler Config laden ---
lstm_cfg = cfg.models.lstm
persist = cfg.model_persistence
model_file = cfg.model_path("lstm")
scaler_file = cfg.model_path("scaler_lstm")

# --- 1. Features vorbereiten ---
# Wir nehmen alle relevanten Informationen für ein "ganzheitliches" Bild
features = cfg.features.model_features
print(f"{MODEL_NAME} nutzt folgende Features: {features}")

window_size = lstm_cfg.window_size

if persist.enabled and os.path.exists(model_file) and os.path.exists(scaler_file):
    # ================================================================
    # MODUS A: Gespeichertes Modell laden (Training überspringen)
    # ================================================================
    model_lstm, scaler, lstm_probs_raw, split = load_lstm_model(
        df=df,
        features=features,
        labels_col=lstm_cfg.labels,
        window_size=window_size,
        train_test_split=lstm_cfg.train_test_split,
        model_file=model_file,
        scaler_file=scaler_file,
    )
else:
    # ================================================================
    # MODUS B: Normal trainieren + Modell speichern
    # ================================================================
    print(f"{MODEL_NAME}: Starte Training...")
    model_lstm, scaler, lstm_probs_raw, split = train_lstm(
        df=df,
        features=features,
        labels_col=lstm_cfg.labels,
        window_size=window_size,
        train_test_split=lstm_cfg.train_test_split,
        units=lstm_cfg.units,
        return_sequences=lstm_cfg.return_sequences,
        dropout=lstm_cfg.dropout,
        dense=lstm_cfg.dense,
        activation=lstm_cfg.activation,
        optimizer=lstm_cfg.optimizer,
        loss=lstm_cfg.loss,
        metrics=lstm_cfg.metrics,
        epochs=lstm_cfg.epochs,
        batch_size=lstm_cfg.batch_size,
        validation_split=lstm_cfg.validation_split,
        verbose=lstm_cfg.verbose,
        model_file=model_file,
        scaler_file=scaler_file,
    )

# --- Wahrscheinlichkeiten und Signal ableiten ---
probs, signal = predict_lstm(
    probs_raw=lstm_probs_raw,
    threshold=lstm_cfg.threshold,
)

# --- Test-DataFrame für Backtesting und Visualisierung vorbereiten ---
# Wir schneiden das df so zu, dass es exakt zu den X_test Daten passt
test_df = df.iloc[split + window_size:].copy()

# Wahrscheinlichkeiten und Signale speichern
test_df[f'{MODEL_NAME}_Prob'] = probs
test_df[f'{MODEL_NAME}_Signal'] = signal

# --- Visualisierung ---
model_color = cfg.color_map.get(MODEL_NAME, 'tab:green')

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(15, 8), sharex=True)

# A. Wahrscheinlichkeiten
ax1.plot(test_df.index, test_df[f'{MODEL_NAME}_Prob'], color=model_color, label=f'{MODEL_NAME} Bear Probability')
ax1.axhline(y=0.5, color='red', linestyle='--', label='Threshold 0.5')
ax1.set_title(f"{MODEL_NAME}: Wahrscheinlichkeit für Bärenmarkt")
ax1.legend()

# B. Signale im Vergleich zum Markov-Label (Grundwahrheit)
ax2.plot(test_df.index, test_df['MSM_Signal'], label='MSM_Univariate Target (Ground Truth)', alpha=0.3, color='gray')
ax2.step(test_df.index, test_df[f'{MODEL_NAME}_Signal'], where='post', label=f'{MODEL_NAME} Signal', color=model_color)
ax2.set_title(f"{MODEL_NAME} Handelssignale (0=Bull, 1=Bear)")
ax2.legend()

plt.tight_layout()
# LSTM-Modell persistieren
plt.savefig(cfg.asset_path("lstm_model"), dpi=300, bbox_inches='tight')
plt.show()

print(test_df)

# --- Wir wechseln in diesem Schritt von df auf test_df da sich der Beobachtungszeitraum eingrenzt ---

# --- Sanity Check ---
validate_regime_signal(test_df, MODEL_NAME)

In [ ]:
# =============================================================================
# 4. Transformer-Netzwerk (Regime Detection via Multi-Head Self-Attention)
# Typ: Supervised (Labels von MS_Univariate)
# Beschreibung: Transformer-Encoder mit Positional Encoding für zeitreihenbasierte
#               Regime-Klassifikation. Testet Hypothese H2 (Attention > Econometric MSM).
# Config-Key: models.transformer
# =============================================================================

import matplotlib.pyplot as plt
import os

# --- Transformer aus src/ ---
from src.models.transformer import train_transformer, load_transformer_model, predict_transformer

MODEL_NAME = "Transformer"

# --- 0. Hyperparameter aus zentraler Config laden ---
t_cfg = cfg.models.transformer
persist = cfg.model_persistence
model_file = cfg.model_path("transformer")
scaler_file = cfg.model_path("scaler_transformer")

# --- 1. Features vorbereiten ---
features = cfg.features.model_features
print(f"Transformer nutzt folgende Features: {features}")

window_size_t = t_cfg.window_size

if persist.enabled and os.path.exists(model_file) and os.path.exists(scaler_file):
    # ================================================================
    # MODUS A: Gespeichertes Modell laden (Training überspringen)
    # ================================================================
    model_transformer, scaler_transformer, transformer_probs_raw, split_t = load_transformer_model(
        df=df,
        features=features,
        labels_col=t_cfg.labels,
        window_size=window_size_t,
        train_test_split=t_cfg.train_test_split,
        d_model=t_cfg.d_model,
        n_heads=t_cfg.n_heads,
        n_layers=t_cfg.n_layers,
        dim_feedforward=t_cfg.dim_feedforward,
        dropout=t_cfg.dropout,
        model_file=model_file,
        scaler_file=scaler_file,
    )
else:
    # ================================================================
    # MODUS B: Normal trainieren + Modell speichern
    # ================================================================
    print(f"{MODEL_NAME}: Starte Training...")
    model_transformer, scaler_transformer, transformer_probs_raw, split_t = train_transformer(
        df=df,
        features=features,
        labels_col=t_cfg.labels,
        window_size=window_size_t,
        train_test_split=t_cfg.train_test_split,
        d_model=t_cfg.d_model,
        n_heads=t_cfg.n_heads,
        n_layers=t_cfg.n_layers,
        dim_feedforward=t_cfg.dim_feedforward,
        dropout=t_cfg.dropout,
        learning_rate=t_cfg.learning_rate,
        epochs=t_cfg.epochs,
        batch_size=t_cfg.batch_size,
        validation_split=t_cfg.validation_split,
        verbose=t_cfg.verbose,
        model_file=model_file,
        scaler_file=scaler_file,
    )

# --- Wahrscheinlichkeiten und Signal ableiten ---
probs, signal = predict_transformer(
    probs_raw=transformer_probs_raw,
    threshold=t_cfg.threshold,
)

# --- Ergebnisse in test_df schreiben ---
test_df[f'{MODEL_NAME}_Prob'] = probs
test_df[f'{MODEL_NAME}_Signal'] = signal

# --- Visualisierung ---
model_color = cfg.color_map.get(MODEL_NAME, 'darkorange')

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(15, 8), sharex=True)

ax1.plot(test_df.index, test_df[f'{MODEL_NAME}_Prob'], color=model_color, label='Transformer Bear Probability')
ax1.axhline(y=0.5, color='red', linestyle='--', label='Threshold 0.5')
ax1.set_title("Transformer: Wahrscheinlichkeit für Bärenmarkt")
ax1.legend()

ax2.plot(test_df.index, test_df['MSM_Signal'], label='MSM_Univariate Target (Ground Truth)', alpha=0.3, color='gray')
ax2.step(test_df.index, test_df[f'{MODEL_NAME}_Signal'], where='post', label='Transformer Signal', color=model_color)
ax2.set_title("Transformer Handelssignale (0=Bull, 1=Bear)")
ax2.legend()

plt.tight_layout()
plt.savefig(cfg.asset_path("transformer_model"), dpi=300, bbox_inches='tight')
plt.show()

# --- Sanity Check ---
validate_regime_signal(test_df, MODEL_NAME)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# 1. Dynamische Identifikation der Modelle
# Wir suchen alle Spalten, die auf _Signal enden, um die Modellnamen zu extrahieren
model_names = [col.rsplit('_', 1)[0] for col in test_df.columns if col.endswith('_Signal')]

# 2. Farbschema definieren (aus config-file)
color_map = cfg.color_map
# Fallback für neue Modelle, die noch nicht in der Map sind
default_colors = plt.rcParams['axes.prop_cycle'].by_key()['color']

# 3. Plot erstellen
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 10), sharex=True)

for i, model in enumerate(model_names):
    # Farbe bestimmen
    color = color_map.get(model, default_colors[i % len(default_colors)])
    
    # Linienstil-Logik
    ls = '-'
    lw = 1.5
    alpha = 0.5
    
    # --- Plot 1: Wahrscheinlichkeiten ---
    prob_col = f"{model}_Prob"
    if prob_col in test_df.columns:
        ax1.plot(test_df.index, test_df[prob_col], 
                 label=f"{model.replace('_', ' ')} Prob", 
                 color=color, linestyle=ls, alpha=alpha, linewidth=lw)

    # --- Plot 2: Signale ---
    sig_col = f"{model}_Signal"
    ax2.step(test_df.index, test_df[sig_col], 
             where='post', label=f"{model.replace('_', ' ')} Signal", 
             color=color, linestyle=ls, alpha=alpha, linewidth=lw)

# --- Ax1 Styling ---
ax1.axhline(y=0.5, color='black', linestyle=':', alpha=0.5, label='Schwelle 0.5')
ax1.set_title("Dynamischer Vergleich der Regime-Wahrscheinlichkeiten (Test-Set)")
ax1.set_ylabel("Wahrscheinlichkeit (Bear)")
ax1.legend(loc='upper left', fontsize='small', ncol=2)
ax1.grid(alpha=0.2)
ax1.set_ylim(-0.05, 1.05)

# --- Ax2 Styling ---
ax2.set_title("Dynamischer Vergleich der Handelssignale (0 = Bull, 1 = Bear)")
ax2.set_ylabel("Signal-Status")
ax2.set_yticks([0, 1])
ax2.set_yticklabels(['Bulle (0)', 'Bär (1)'])
ax2.legend(loc='upper left', fontsize='small', ncol=2)
ax2.grid(alpha=0.2)
ax2.set_ylim(-0.1, 1.1)

# Layout optimieren
plt.tight_layout()
# Regime Comparison persistieren
plt.savefig(cfg.asset_path("regime_comparison"), dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
output_path = cfg.data_path("test_data")

# Speichern als Parquet
test_df.to_parquet(output_path)

print(f"Dataframe erfolgreich unter {output_path} gespeichert.")